In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler,OneHotEncoder,LabelEncoder
from sklearn.model_selection import train_test_split
import pickle


In [3]:
## loading the data 
data = pd.read_csv('predictive_maintenance_dataset (1).csv')
data.head(20)

,timestamp,machine_id,vibration,acoustic,temperature,current,IMF_1,IMF_2,IMF_3,label
0,2024-07-01 08:00:00,M01,0.822,0.645,66.85,13.04,0.196,0.033,0.000,0
1,2024-07-01 08:01:00,M01,1.398,0.834,76.20,15.08,0.345,0.132,0.001,1
2,2024-07-01 08:02:00,M01,0.856,0.590,67.03,12.30,0.187,0.017,0.002,0
3,2024-07-01 08:03:00,M01,0.793,0.544,65.04,11.69,0.196,-0.060,0.003,0
4,2024-07-01 08:04:00,M01,1.279,0.721,78.19,14.84,0.330,-0.115,0.004,1
5,2024-07-01 08:05:00,M01,0.782,0.655,61.95,11.56,0.164,-0.081,0.005,0
6,2024-07-01 08:06:00,M01,0.837,0.564,64.52,10.98,0.238,-0.095,0.006,0
7,2024-07-01 08:07:00,M01,0.725,0.652,62.87,11.89,0.214,-0.076,0.007,0
8,2024-07-01 08:08:00,M01,0.803,0.702,64.51,11.19,0.145,-0.004,0.008,0
9,2024-07-01 08:09:00,M01,0.779,0.539,61.30,12.51,0.103,-0.025,0.009,0


In [4]:
## dropping the irrelevant data
data = data.drop(['timestamp','machine_id'], axis=1)
data.head()

,vibration,acoustic,temperature,current,IMF_1,IMF_2,IMF_3,label
0,0.822,0.645,66.85,13.04,0.196,0.033,0.000,0
1,1.398,0.834,76.20,15.08,0.345,0.132,0.001,1
2,0.856,0.590,67.03,12.30,0.187,0.017,0.002,0
3,0.793,0.544,65.04,11.69,0.196,-0.060,0.003,0
4,1.279,0.721,78.19,14.84,0.330,-0.115,0.004,1


In [5]:
## now dividing the data into independant and dependant feature
X = data.drop('label', axis=1)
y = data['label']

## now seprate the data for train and test split
X_train,X_test,y_train,y_test = train_test_split(X,y, test_size=0.2, random_state=42 )

## now scaling the data 
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [7]:
X_test

array([[-0.24264252, -0.51644648, -0.14582408, ..., -0.29821738,
        -1.41545947,  1.37375188],
       [-0.55887962, -1.05511833, -0.03190847, ..., -1.22679843,
         0.07563831,  0.73126967],
       [ 0.26480772,  0.23583662, -0.81591592, ..., -1.24465576,
         0.4346063 , -1.11237491],
       ...,
       [ 0.02211413,  0.1893994 , -0.1748614 , ...,  0.59464902,
         1.6357684 , -1.36378099],
       [ 0.02946848, -0.75792006,  0.02616616, ..., -1.45894369,
        -0.82178165,  1.34581787],
       [-0.30883168, -0.3585599 ,  0.03286707, ..., -0.24464539,
        -0.73894288,  1.37375188]])

In [8]:
## now creating pickle file for scalar so any value comes we first scale that value
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler, file)

In [9]:
data

,vibration,acoustic,temperature,current,IMF_1,IMF_2,IMF_3,label
0,0.822,0.645,66.85,13.04,0.196,0.033,0.000,0
1,1.398,0.834,76.20,15.08,0.345,0.132,0.001,1
2,0.856,0.590,67.03,12.30,0.187,0.017,0.002,0
3,0.793,0.544,65.04,11.69,0.196,-0.060,0.003,0
4,1.279,0.721,78.19,14.84,0.330,-0.115,0.004,1
...,...,...,...,...,...,...,...,...
1795,1.163,0.889,78.43,14.41,0.261,0.041,-0.031,1
1796,0.838,0.545,64.28,12.08,0.146,-0.062,-0.030,0
1797,0.786,0.593,65.44,12.23,0.110,0.038,-0.029,0
1798,0.767,0.550,62.01,11.35,0.225,0.106,-0.028,0


In [11]:
### ANN implementation
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense


In [13]:
## lets build our ANN model
model = Sequential([

Dense(64, activation='relu', input_shape = (X_train.shape[1],)),
Dense(32, activation='relu'),
Dense(1, activation='sigmoid')




]



)

In [14]:
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_3 (Dense)             (None, 64)                512       
                                                                 
 dense_4 (Dense)             (None, 32)                2080      
                                                                 
 dense_5 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2625 (10.25 KB)
Trainable params: 2625 (10.25 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [16]:
## compiling the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [18]:
from tensorflow.keras.callbacks import EarlyStopping

early_stooping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [19]:
## now lets train the model
history = model.fit(
    X_train,y_train, validation_data=(X_test,y_test), epochs = 100,
    callbacks = [early_stooping_callback]
)

Epoch 1/100


45/45 [==============================] - 1s 4ms/step - loss: 0.3501 - accuracy: 0.9715 - val_loss: 0.1229 - val_accuracy: 1.0000
Epoch 2/100
45/45 [==============================] - 0s 2ms/step - loss: 0.0600 - accuracy: 1.0000 - val_loss: 0.0229 - val_accuracy: 1.0000
Epoch 3/100
45/45 [==============================] - 0s 1ms/step - loss: 0.0158 - accuracy: 1.0000 - val_loss: 0.0094 - val_accuracy: 1.0000
Epoch 4/100
45/45 [==============================] - 0s 1ms/step - loss: 0.0077 - accuracy: 1.0000 - val_loss: 0.0052 - val_accuracy: 1.0000
Epoch 5/100
45/45 [==============================] - 0s 1ms/step - loss: 0.0047 - accuracy: 1.0000 - val_loss: 0.0034 - val_accuracy: 1.0000
Epoch 6/100
45/45 [==============================] - 0s 1ms/step - loss: 0.0032 - accuracy: 1.0000 - val_loss: 0.0024 - val_accuracy: 1.0000
Epoch 7/100
45/45 [==============================] - 0s 1ms/step - loss: 0.0024 - accuracy: 1.0000 - val_loss: 0.0018 - val_accuracy: 1.0000
Epoch 8/100

In [20]:
print(data['label'].value_counts())

label
0    1598
1     202
Name: count, dtype: int64


In [21]:
print("Train:")
print(y_train.value_counts())

print("\nTest:")
print(y_test.value_counts())

Train:
label
0    1279
1     161
Name: count, dtype: int64

Test:
label
0    319
1     41
Name: count, dtype: int64


In [22]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

12/12 [==============================] - 0s 752us/step
[[319   0]
 [  0  41]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       319
           1       1.00      1.00      1.00        41

    accuracy                           1.00       360
   macro avg       1.00      1.00      1.00       360
weighted avg       1.00      1.00      1.00       360



In [24]:
print(data.groupby('label').mean(numeric_only=True))

       vibration  acoustic  temperature    current     IMF_1     IMF_2  \
label                                                                    
0       0.798652  0.600558    64.959819  11.989587  0.160327 -0.000345   
1       1.196460  0.897644    77.398564  15.010495  0.235272  0.003371   

          IMF_3  
label            
0      0.000986  
1     -0.001797  


In [26]:
## save the model
model.save('model.h5')

c:\Users\Ayan\Desktop\ANN_Project_2\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
